# 🧪 实验四：模型评测与对话测试

## 学习目标
- 加载微调后的模型
- 测试模型的实际对话效果

## 1. 加载微调后的模型（NPU + FlashAttention 加速）

> **硬件说明**：本实验基于 **Ascend NPU** 运行。模型加载时继续使用 `attn_implementation="sdpa"` 保持 FlashAttention 加速。
> 
> 微调后的模型保存在 `./sft_output` 目录中，包含 LoRA 适配器权重和分词器文件。

In [ ]:
import torch
import torch_npu  # noqa: F401 — 注册 Ascend NPU 后端
torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast

MODEL_PATH = "./sft_output"  # 训练好的模型路径

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=f"{MODEL_PATH}/tokenizer.json",
    pad_token="</s>",
    eos_token="</s>",
    bos_token="<s>"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa"  # 🔥 保持 SDPA（FlashAttention）加速
)

print(f"微调模型已加载: {model.device}")

---

### 🔥 为什么推理阶段也要保持 SDPA？

在推理（生成）阶段，FlashAttention 同样重要：

1. **加速生成**：每个 token 生成时都需要计算注意力，SDPA 通过 **Tiling + Online Softmax** 减少 HBM 访问，让这一步快 3-8x
2. **支持长上下文**：本实验的 `chat()` 函数中，输入序列最长 1024 tokens，SDPA 的线性显存特性让长序列生成更稳定
3. **与 LoRA 权重兼容**：LoRA 适配器注入的是低秩矩阵，不影响注意力计算方式，SDPA 对 LoRA 模型完全透明

> **注意**：推理时 `model.generate()` 内部使用 `use_cache=True`（KV Cache），这与 SDPA 兼容——SDPA 加速的是每个解码步的注意力计算，而 KV Cache 减少的是重复计算，两者叠加效果更好。

### 参考文章

- [AIInfraGuide — FlashAttention V1 详解](https://caomaolufei.github.io/AIInfraGuide/guides/模块二-cuda编程与算子优化/61-flashattention-v1详解/)
- [AIInfraGuide — FlashAttention V2 详解](https://caomaolufei.github.io/AIInfraGuide/guides/模块二-cuda编程与算子优化/62-flashattention-v2详解/)

---

## 2. 单轮对话测试

In [ ]:
def chat(query, max_new_tokens=256):
    """推理函数：用对话格式包装，只生成一轮回答"""
    prompt = f"<|user|>\n{query}\n<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
    input_len = inputs["input_ids"].shape[1]
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    new_tokens = outputs[0][input_len:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=False)
    for stop in ["<|end|>", "<|user|>", "<|assistant|>"]:
        pos = answer.find(stop)
        if pos != -1:
            answer = answer[:pos]
    return answer.strip()

In [ ]:
test_questions = [
    "我总是焦虑不安，压力大得喘不过气，怎么办？",
    "每天提不起劲，觉得自己一无是处，是不是抑郁了？",
    "我害怕在很多人面前说话，紧张到说不出话，这是病吗？",
    "最近总是失眠，躺在床上脑子停不下来，怎么办？",
]

for q in test_questions:
    print(f"输入：{q}")
    print(f"输出：{chat(q)}")
    print("-" * 60)

## 小结

✅ 成功加载了微调后的模型（LoRA 适配器 + SDPA FlashAttention 加速）
✅ 测试了单轮对话，对比了心理健康领域多个问题的回答质量
✅ 推理阶段 SDPA 持续生效，加速每个 token 的生成

下一节将进行多轮交互式对话测试。

## 课后练习

1. (单选题) do_sample=False 且 temperature=0.8 时，生成方式为？
   - A. 贪心解码，temperature 不生效
   - B. 随机采样
   - C. top-k 采样
   - D. 温度采样

2. (单选题) 输入长度 900、max_new_tokens=128、模型窗口 1024，会发生？
   - A. 总长度 1028 超出窗口，可能报错或截断
   - B. 一定正常
   - C. 自动忽略输入
   - D. 只生成 124 tokens

3. (多选题) 降低回答重复的手段包括？
   - A. repetition_penalty
   - B. no_repeat_ngram_size
   - C. 合适的 temperature/top_p
   - D. 增大 max_new_tokens

4. (多选题) 心理咨询场景对话评估维度包括？
   - A. 专业准确性
   - B. 共情表达
   - C. 建议可执行性
   - D. 安全性

5. (判断题) torch.no_grad() 下调用 generate 可避免保存反向图，降低显存。

6. (判断题) do_sample=False 时，temperature 仍会改变输出分布。

7. (填空题) top_p=0.9 表示从累积概率达到 0.9 的最小候选集合中采样，称为 ____。

8. (填空题) generate 中控制新增 token 数量的是 ____，控制停止的是 ____。

9. (简答题) 为什么回答截断既要处理 <|end|> 也要处理 <|user|>？

10. (简答题) 如何设计可复现的对话评估流程？

11. (代码设计题) 编写 chat_infer(model, tokenizer, prompt, max_new_tokens, do_sample, temperature, top_p)，返回清理后的回答。

12. (单选题) 模型持续输出同一句话，最直接有效的参数是？
   - A. repetition_penalty
   - B. max_new_tokens
   - C. lora_alpha
   - D. batch_size

13. (多选题) 安全性评估应覆盖？
   - A. 有害请求拒绝
   - B. 隐私边界
   - C. 诱导性 prompt
   - D. 回答长度

14. (判断题) temperature 越高，回答质量一定越高。

15. (简答题) 为心理咨询机器人设计 4 维度评分表，并说明每个维度的评分标准。

> 参考答案见 answer/05.05_evaluation_and_dialogue_test_answer.ipynb。